In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os


In [2]:
sys.path.append(os.path.abspath(".."))

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from src.data_utils import (load_data)
from src.cleaning_utils import (save_cleaned_data)
from src.feature_utils import (
basic_clean,
fit_missing_values,
apply_missing_values,
fit_geo_imputer,
apply_geo_imputer,
fit_log_transform,
apply_log_transform,
fit_value_replacement,
apply_value_replacement,
fit_outlier_caps,
apply_outlier_caps,
fit_rare_categories,
apply_rare_categories,
create_regular_features,
create_binary_features,
target_encode_lga


)

In [4]:
X_train = load_data("../data/X_train.csv")
y_train = load_data("../data/y_train.csv")

df_raw = X_train.merge(y_train, on="id")

DataFrame successfully loaded.
Shape: (59400, 40)
DataFrame successfully loaded.
Shape: (59400, 2)


In [5]:
df = basic_clean(df_raw)

Dropped columns: ['id', 'wpt_name', 'num_private', 'subvillage', 'region_code', 'district_code', 'ward', 'recorded_by', 'scheme_name', 'extraction_type_group', 'extraction_type_class', 'payment_type', 'water_quality', 'quantity_group', 'source', 'waterpoint_type_group']
 Placeholder values replaced:
{'funder': 781, 'installer': 781, 'management': 561, 'management_group': 561, 'payment': 8157, 'quality_group': 1876, 'quantity': 789, 'source_class': 278}
Converted to category: []
Converted to datetime: ['date_recorded']
Dropped 592 duplicate rows


In [6]:
X = df.drop(columns=["status_group"])
y = df["status_group"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
missing_stats = fit_missing_values(X_train)

X_train = apply_missing_values(X_train, missing_stats)
X_test = apply_missing_values(X_test, missing_stats)

In [8]:
rare_stats = fit_rare_categories(X_train, cols=[
    "extraction_type",
    "waterpoint_type",
    "installer",
    "funder",
    "scheme_management",
    "management",
    "water_quality",
    "quality_group",
    "quantity",
    "payment",
    "payment_type",
    "source",
    "source_type",
    "source_class"])
X_train = apply_rare_categories(X_train, rare_stats)
X_test = apply_rare_categories(X_test, rare_stats)


In [9]:
stats = fit_value_replacement(
    X_train,
    cols=["gps_height", "population"],
    treat_zero_as_nan=["gps_height", "population"],
    treat_negative_as_nan=["gps_height"],
    group_cols=["lga"]
)

X_train = apply_value_replacement(
    X_train,
    stats,
    cols=["gps_height", "population"],
    treat_zero_as_nan=["gps_height", "population"],
    treat_negative_as_nan=["gps_height"],
    group_cols=["lga"]
)

X_test = apply_value_replacement(
    X_test,
    stats,
    cols=["gps_height", "population"],
    treat_zero_as_nan=["gps_height", "population"],
    treat_negative_as_nan=["gps_height"],
    group_cols=["lga"]
)

In [10]:
num_cols = df.select_dtypes(include=["number"]).columns

log_cols = fit_log_transform(
    X_train,
    cols=num_cols,  # optional
    skew_threshold=1.0
)

print("Skewed cols are: ", log_cols)

X_train = apply_log_transform(X_train, log_cols)
X_test  = apply_log_transform(X_test, log_cols)

Skewed cols are:  ['amount_tsh', 'population']


In [11]:
geo_stats = fit_geo_imputer(
    X_train,
    lat_col="latitude",
    lon_col="longitude"
)

X_train = apply_geo_imputer(
    X_train,
    geo_stats,
    lat_col="latitude",
    lon_col="longitude"
)

X_test = apply_geo_imputer(
    X_test,
    geo_stats,
    lat_col="latitude",
    lon_col="longitude"
)

In [12]:
X_train.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
amount_tsh,47046.0,NaN,NaN,NaN,329.399372,0.0,0.0,0.0,25.0,350000.0,3248.682978
date_recorded,47046,NaN,NaN,NaN,2012-03-27 16:34:04.024996,2004-01-07 00:00:00,2011-03-31 00:00:00,2012-10-09 00:00:00,2013-02-09 00:00:00,2013-12-03 00:00:00,NaN
funder,47046,26,others,19051,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gps_height,47046.0,NaN,NaN,NaN,1077.722389,1.0,856.0,1193.0,1347.0,2770.0,503.563466
installer,47046,26,others,18073,NaN,NaN,NaN,NaN,NaN,NaN,NaN
longitude,47046.0,NaN,NaN,NaN,35.13333,29.676978,33.264693,34.895655,37.18286,40.16164,2.578619
latitude,47046.0,NaN,NaN,NaN,-5.764642,-11.11914,-8.53699,-4.980109,-3.344201,-0.0,2.900039
basin,47046,9,Lake Victoria,7710,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,47046,21,Iringa,4286,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lga,47046,125,Njombe,2036,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
num_cols = X_train.select_dtypes(include=np.number).columns

for col in num_cols:
    print(col, "zeros:", (X_train[col] == 0).sum(), "neg:", (X_train[col] < 0).sum())

amount_tsh zeros: 32805 neg: 0
gps_height zeros: 0 neg: 0
longitude zeros: 0 neg: 0
latitude zeros: 0 neg: 47046
population zeros: 0 neg: 0
construction_year zeros: 0 neg: 0
amount_tsh_log zeros: 32805 neg: 0
population_log zeros: 0 neg: 0


In [14]:
X_train = create_regular_features(X_train)
X_train = create_binary_features(X_train)

In [15]:
X_test = create_regular_features(X_test)
X_test = create_binary_features(X_test)

In [16]:
X_train, X_test, lga_map = target_encode_lga(
    X_train,
    X_test,
    y_train,
    target_col="status_group",
    cat_col="lga"
)

In [17]:
low_card_cols = [
    "basin", "region", "public_meeting", "permit",
    "extraction_type", "management_group", "quality_group",
    "quantity", "source_class", "waterpoint_type", "payment", "source_type"
]

In [18]:
high_card_cols = ["funder", "installer", "scheme_management", "management"]

In [19]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

ohe.fit(X_train[low_card_cols])

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_cate

In [20]:
train_ohe = pd.DataFrame(
    ohe.transform(X_train[low_card_cols]),
    columns=ohe.get_feature_names_out(low_card_cols),
    index=X_train.index
)

In [21]:
test_ohe = pd.DataFrame(
    ohe.transform(X_test[low_card_cols]),
    columns=ohe.get_feature_names_out(low_card_cols),
    index=X_test.index
)

In [22]:
X_train = X_train.drop(columns=low_card_cols).join(train_ohe)
X_test = X_test.drop(columns=low_card_cols).join(test_ohe)

In [23]:
freq_maps = {}

for col in high_card_cols:
    freq_maps[col] = X_train[col].value_counts(normalize=True)

for col in high_card_cols:
    X_train[col + "_freq"] = X_train[col].map(freq_maps[col])

In [24]:
for col in high_card_cols:
    X_test[col + "_freq"] = X_test[col].map(freq_maps[col])

X_test[col + "_freq"] = X_test[col + "_freq"].fillna(0)

In [25]:
X_train = X_train.drop(columns=high_card_cols)
X_test = X_test.drop(columns=high_card_cols)

In [26]:
save_cleaned_data(X_train, "../data/X_train_with_new_features.csv")
save_cleaned_data(X_test, "../data/X_test_with_new_features.csv")
save_cleaned_data(y_train, "../data/y_train_new.csv")
save_cleaned_data(y_test, "../data/y_test_new.csv")

Saved cleaned data to: ../data/X_train_with_new_features.csv
Saved cleaned data to: ../data/X_test_with_new_features.csv
Saved cleaned data to: ../data/y_train_new.csv
Saved cleaned data to: ../data/y_test_new.csv


In [27]:
X_train.describe(include="all").T

,count,mean,std,min,25%,50%,75%,max
amount_tsh,47046.0,329.399372,3248.682978,0.000000,0.000000,0.000000,25.000000,3.500000e+05
gps_height,47046.0,1077.722389,503.563466,1.000000,856.000000,1193.000000,1347.000000,2.770000e+03
longitude,47046.0,35.133330,2.578619,29.676978,33.264693,34.895655,37.182860,4.016164e+01
latitude,47046.0,-5.764642,2.900039,-11.119140,-8.536990,-4.980109,-3.344201,-2.000000e-08
population,47046.0,243.531831,467.981454,1.000000,96.000000,150.000000,250.000000,3.050000e+04
...,...,...,...,...,...,...,...,...
source_type_spring,47046.0,0.289143,0.453369,0.000000,0.000000,0.000000,1.000000,1.000000e+00
funder_freq,47046.0,0.223407,0.168312,0.006462,0.022467,0.226183,0.404944,4.049441e-01
installer_freq,47046.0,0.283907,0.156794,0.004719,0.029057,0.364516,0.384156,3.841559e-01
scheme_management_freq,47046.0,0.486284,0.294813,0.001233,0.083195,0.685691,0.685691,6.856906e-01


In [28]:
print(X_train.columns.tolist())

['amount_tsh', 'gps_height', 'longitude', 'latitude', 'population', 'construction_year', 'amount_tsh_log', 'population_log', 'recorded_year', 'recorded_month', 'pump_age', 'pump_is_new', 'pump_age_band', 'height_band', 'quantity_score', 'is_paid_water', 'is_water_safe', 'lga_te', 'basin_Internal', 'basin_Lake Nyasa', 'basin_Lake Rukwa', 'basin_Lake Tanganyika', 'basin_Lake Victoria', 'basin_Pangani', 'basin_Rufiji', 'basin_Ruvuma / Southern Coast', 'basin_Wami / Ruvu', 'region_Arusha', 'region_Dar es Salaam', 'region_Dodoma', 'region_Iringa', 'region_Kagera', 'region_Kigoma', 'region_Kilimanjaro', 'region_Lindi', 'region_Manyara', 'region_Mara', 'region_Mbeya', 'region_Morogoro', 'region_Mtwara', 'region_Mwanza', 'region_Pwani', 'region_Rukwa', 'region_Ruvuma', 'region_Shinyanga', 'region_Singida', 'region_Tabora', 'region_Tanga', 'public_meeting_False', 'public_meeting_True', 'permit_False', 'permit_True', 'extraction_type_afridev', 'extraction_type_cemo', 'extraction_type_climax', 'e

In [29]:
X_train.info()


<class 'pandas.DataFrame'>
Index: 47046 entries, 8092 to 27966
Columns: 109 entries, amount_tsh to management_freq
dtypes: float64(104), int32(2), int64(3)
memory usage: 40.1 MB
